In [52]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
from itertools import product

In [2]:
# Load the data set
dataset = pd.read_csv("insurance_pre.csv")

In [3]:
# Show the original data
print(f"Original Data:\n{dataset}")

Original Data:
      age     sex     bmi  children smoker      charges
0      19  female  27.900         0    yes  16884.92400
1      18    male  33.770         1     no   1725.55230
2      28    male  33.000         3     no   4449.46200
3      33    male  22.705         0     no  21984.47061
4      32    male  28.880         0     no   3866.85520
...   ...     ...     ...       ...    ...          ...
1333   50    male  30.970         3     no  10600.54830
1334   18  female  31.920         0     no   2205.98080
1335   18  female  36.850         0     no   1629.83350
1336   21  female  25.800         0     no   2007.94500
1337   61  female  29.070         0    yes  29141.36030

[1338 rows x 6 columns]


In [8]:
# One-Hot Encoding
dataset = pd.get_dummies(dataset, drop_first = 1)

In [10]:
# Show the One-Hot data
print(f"One-Hot Encoding:\n{dataset}")

One-Hot Encoding:
      age     bmi  children      charges  sex_male  smoker_yes
0      19  27.900         0  16884.92400     False        True
1      18  33.770         1   1725.55230      True       False
2      28  33.000         3   4449.46200      True       False
3      33  22.705         0  21984.47061      True       False
4      32  28.880         0   3866.85520      True       False
...   ...     ...       ...          ...       ...         ...
1333   50  30.970         3  10600.54830      True       False
1334   18  31.920         0   2205.98080     False       False
1335   18  36.850         0   1629.83350     False       False
1336   21  25.800         0   2007.94500     False       False
1337   61  29.070         0  29141.36030     False        True

[1338 rows x 6 columns]


In [12]:
# Display the data set columns
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [14]:
# Future and Target
x = dataset.drop("charges", axis = 1)
y = dataset[["charges"]]

In [16]:
print(f"Independent value:\n{x}")
print(f"Dependent value:\n{y}")

Independent value:
      age     bmi  children  sex_male  smoker_yes
0      19  27.900         0     False        True
1      18  33.770         1      True       False
2      28  33.000         3      True       False
3      33  22.705         0      True       False
4      32  28.880         0      True       False
...   ...     ...       ...       ...         ...
1333   50  30.970         3      True       False
1334   18  31.920         0     False       False
1335   18  36.850         0     False       False
1336   21  25.800         0     False       False
1337   61  29.070         0     False        True

[1338 rows x 5 columns]
Dependent value:
          charges
0     16884.92400
1      1725.55230
2      4449.46200
3     21984.47061
4      3866.85520
...           ...
1333  10600.54830
1334   2205.98080
1335   1629.83350
1336   2007.94500
1337  29141.36030

[1338 rows x 1 columns]


In [18]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 42)

In [150]:
def generate_param_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    return [dict(zip(keys, combo)) for combo in product(*values)]
    
param_linear_regression = {
    'copy_X': [True, False],
    'fit_intercept': [True, False],
    'n_jobs': [None, 1, -1],
    'positive': [True, False]
}
param_support_vector = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 1],
    'gamma': ['scale', 'auto']
}
param_decision_tree = {
    'criterion': ['squared_error', 'absolute_error'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
param_random_forest = {
    'n_estimators': [100, 200, 300],
    'criterion': ['squared_error', 'absolute_error'],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
param_ada_boost = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1.0],
    'loss': ['linear', 'square', 'exponential']
}
param_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 1, 5],
    'reg_alpha': [0, 0.5, 1],
    'reg_lambda': [0.5, 1, 2]
}


combinations = generate_param_combinations(param_xgb)

print(f"Total combinations: {len(combinations)}")
for combo in combinations:
    print(combo)

Total combinations: 6561
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 0.5}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 1}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 2}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda': 0.5}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda': 1}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda': 2}
{'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample

In [152]:
# 1. Define param sets (you can define multiple options per model)
params = {
    #"LinearRegression": generate_param_combinations(param_linear_regression),
    #"SupportVectorMachine": generate_param_combinations(param_support_vector),
    #"DecisionTreeRegressor": generate_param_combinations(param_decision_tree),
    #"RandomForestRegressor": generate_param_combinations(param_random_forest),
    #"AdaBoostRegressor": generate_param_combinations(param_ada_boost),
    "XGBRegressor": generate_param_combinations(param_xgb),
    # Add more models here with lists of param dicts if needed
}

In [154]:
models = {
    "LinearRegression": LinearRegression,
    "SupportVectorMachine": SVR,
    "DecisionTreeRegressor": DecisionTreeRegressor,
    "RandomForestRegressor": RandomForestRegressor,
    "AdaBoostRegressor": AdaBoostRegressor,
    "XGBRegressor": XGBRegressor
    
}

In [156]:
results = {}
best_model = None
best_score = float('-inf')
best_model_name = ""


In [158]:
for name, model_class in models.items():
    if name in params:
        print(f"Model {name} has custom parameters:")
        for i, param_set in enumerate(params[name]):
            model = model_class(**param_set)
            model.fit(X_train, y_train)
            print(f"Params {i+1}: {param_set}")
            y_pred = model.predict(X_test)
            r2_result = r2_score(y_test, y_pred)
            results[f"{name}_params_{i+1}:{param_set}"] = r2_result
            
            # Track best model
            if r2_result > best_score:
                best_score = r2_result
                best_model = model
                best_model_name = f"{name}_params_{i+1}"
            # Save the the model
            filename = f"{name}.sav"
            pickle.dump(best_model, open(filename, "wb"))

    

Model XGBRegressor has custom parameters:
Params 1: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 0.5}
Params 2: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 1}
Params 3: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 2}
Params 4: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda': 0.5}
Params 5: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda': 1}
Params 6: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'gamma': 0, 'reg_alpha': 0.5, 'reg_lambda

In [159]:
best_score

0.8668158054351807

In [160]:
for value in results.values():
    print(value)

0.5210423469543457
0.5184874534606934
0.5132037997245789
0.5210421085357666
0.5184872150421143
0.5132036805152893
0.5210418701171875
0.5184869766235352
0.513203501701355
0.5210423469543457
0.5184874534606934
0.5132037997245789
0.5210421085357666
0.5184872150421143
0.5132036805152893
0.5210418701171875
0.5184869766235352
0.513203501701355
0.5210423469543457
0.5184874534606934
0.5132037997245789
0.5210421085357666
0.5184872150421143
0.5132036805152893
0.5210418701171875
0.5184869766235352
0.513203501701355
0.6452429294586182
0.6429697275161743
0.635920524597168
0.6452425718307495
0.6429696083068848
0.6359202265739441
0.6452423334121704
0.6429693698883057
0.635919988155365
0.6452429294586182
0.6429697275161743
0.635920524597168
0.6452425718307495
0.6429696083068848
0.6359202265739441
0.6452423334121704
0.6429693698883057
0.635919988155365
0.6452429294586182
0.6429697275161743
0.635920524597168
0.6452425718307495
0.6429696083068848
0.6359202265739441
0.6452423334121704
0.6429693698883057
0